# Stage 1: Environment Setup and Loading Dataset

[1] Installing Dependencies:-

In [ ]:
!pip install transformers datasets scikit-learn groq pandas numpy torch -q

print("All libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.9 MB/s eta 0:00:00
All libraries installed successfully.


[2] Verifying GPU:

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM available  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

PyTorch version : 2.10.0+cu128
GPU available   : True
GPU name        : Tesla T4
VRAM available  : 15.6 GB


[3] Groq API Setup

In [ ]:
import os

GROQ_API_KEY = "gsk_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # Replace with your actual API key

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

test_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Reply with just the word: CONNECTED"}],
    max_tokens=10
)

print(f"Groq API status: {test_response.choices[0].message.content.strip()}")

Groq API status: CONNECTED


[4]  Load Financial PhraseBank Dataset

In [ ]:
import pandas as pd

# Load the manually uploaded file
df = pd.read_csv("/content/all-data.csv",
                 encoding="latin-1",
                 header=None,
                 names=["sentiment", "text"])

# Standardise labels
df["sentiment"] = df["sentiment"].str.lower().str.strip()

# Filter to "allagree" equivalent — high confidence samples only
# (this version has all samples; we keep all for maximum training data)
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["sentiment_label"] = df["sentiment"].map(label_map)

# Drop any rows with unmapped labels
df = df.dropna(subset=["sentiment_label"]).reset_index(drop=True)
df["sentiment_label"] = df["sentiment_label"].astype(int)

print(f"Dataset loaded successfully.")
print(f"Total samples : {len(df)}")
print(f"\nClass distribution:")
print(df["sentiment"].value_counts())

Dataset loaded successfully.
Total samples : 4846

Class distribution:
sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64


[5] Analyse Class Distribution

In [ ]:
import matplotlib.pyplot as plt

# Class distribution
distribution = df["sentiment"].value_counts()
percentages  = df["sentiment"].value_counts(normalize=True) * 100

print("=" * 40)
print("CLASS DISTRIBUTION ANALYSIS")
print("=" * 40)
for label in ["positive", "neutral", "negative"]:
    count = distribution[label]
    pct   = percentages[label]
    bar   = "█" * int(pct / 2)
    print(f"{label:>10} : {count:>5} samples ({pct:.1f}%)  {bar}")

print("=" * 40)
print(f"\nImbalance ratio (neutral:negative) = {distribution['neutral']/distribution['negative']:.1f}x")
print("→ Class-weighted loss will be needed during fine-tuning.")

CLASS DISTRIBUTION ANALYSIS
  positive :  1363 samples (28.1%)  ██████████████
   neutral :  2879 samples (59.4%)  █████████████████████████████
  negative :   604 samples (12.5%)  ██████

Imbalance ratio (neutral:negative) = 4.8x
→ Class-weighted loss will be needed during fine-tuning.


[6] Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]   # preserves class proportions in each split
)

# Second split: temp → 50% val, 50% test (i.e., 10% + 10% of total)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["sentiment"]
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train samples : {len(train_df)}")
print(f"Val   samples : {len(val_df)}")
print(f"Test  samples : {len(test_df)}")
print(f"\nTrain class distribution:")
print(train_df["sentiment"].value_counts().to_string())

Train samples : 3876
Val   samples : 485
Test  samples : 485

Train class distribution:
sentiment
neutral     2303
positive    1090
negative     483


[7] Mounting drive and saving dataset

In [ ]:
from google.colab import drive

# Mount Google Drive to persist your data between sessions
drive.mount('/content/drive')

# Create project folder
import os
os.makedirs('/content/drive/MyDrive/KD_Project', exist_ok=True)

# Save all splits
train_df.to_csv('/content/drive/MyDrive/KD_Project/train.csv', index=False)
val_df.to_csv('/content/drive/MyDrive/KD_Project/val.csv',   index=False)
test_df.to_csv('/content/drive/MyDrive/KD_Project/test.csv',  index=False)
df.to_csv('/content/drive/MyDrive/KD_Project/full_dataset.csv', index=False)

print("All splits saved to Google Drive successfully.")
print("\nFiles saved:")
print("  /content/drive/MyDrive/KD_Project/train.csv")
print("  /content/drive/MyDrive/KD_Project/val.csv")
print("  /content/drive/MyDrive/KD_Project/test.csv")
print("  /content/drive/MyDrive/KD_Project/full_dataset.csv")

Mounted at /content/drive
All splits saved to Google Drive successfully.

Files saved:
  /content/drive/MyDrive/KD_Project/train.csv
  /content/drive/MyDrive/KD_Project/val.csv
  /content/drive/MyDrive/KD_Project/test.csv
  /content/drive/MyDrive/KD_Project/full_dataset.csv
